# BÀI TẬP: E-COMMERCE DATA (ONLINE RETAIL)
**Nguồn:** kaggle.com/datasets/carrie1/ecommerce-data (541,909 dòng)


## Setup

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv'

df = pd.read_csv(csv_path, encoding='ISO-8859-1', on_bad_lines='skip')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [2]:
# TODO
print('Số hàng:', df.shape[0], 'Số cột:', df.shape[1])
print()
print('Columns:',df.columns.tolist())
print()
print('Data types:', df.dtypes)


Số hàng: 541909 Số cột: 8

Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Data types: InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object


## A.2. Missing values & Duplicate data

In [3]:
# TODO
# TODO
missing = df.isnull().sum()
print("Missing values:")
print(missing[missing>0] if missing.sum()>0 else "Không có cột nào thiếu dữ liệu.")
print()
print('Duplicate values:', df.duplicated().sum())

Missing values:
Description      1454
CustomerID     135080
dtype: int64

Duplicate values: 5268


## A.3. Invalid values

In [5]:
# TODO
# TODO
print("Quantity <= 0:", (df['Quantity'] <= 0).sum())
print("UnitPrice <= 0:", (df['UnitPrice'] <= 0).sum())
print()
print("Kết luận: Dữ liệu cần được làm sạch các giá trị <= 0.")

Quantity <= 0: 10624
UnitPrice <= 0: 2517

Kết luận: Dữ liệu cần được làm sạch các giá trị <= 0.


## A.4. Create a new column
Làm sạch dữ liệu (loại Quantity<=0, UnitPrice<=0), tạo cột `Sales` = Quantity * UnitPrice.

In [6]:
# TODO
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['Sales'] = df['Quantity'] * df['UnitPrice']
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [7]:
# TODO
# TODO
for col in ['Quantity', 'UnitPrice', 'Sales']:
    s = df[col].dropna()
    mean, median, mode = s.mean(), s.median(), s.mode()[0]
    print(f"{col.upper()}: Mean={mean:.2f}, Median={median:.2f}, Mode={mode:.2f}")

QUANTITY: Mean=10.54, Median=3.00, Mode=1.00
UNITPRICE: Mean=3.91, Median=2.08, Mode=1.25
SALES: Mean=20.12, Median=9.90, Mode=15.00


## Group 2 — Dispersion

In [8]:
# TODO
# TODO
for col in ['Quantity', 'UnitPrice', 'Sales']:
    s = df[col].dropna()
    range_ = s.max() - s.min()
    std_ = s.std()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    cv = std_ / s.mean()
    print(f"--- {col.upper()} ---")
    print(f"Range={range_:.2f}, Std={std_:.2f}, IQR={iqr:.2f}, CV={cv:.2f}\n")

--- QUANTITY ---
Range=80994.00, Std=155.52, IQR=9.00, CV=14.75

--- UNITPRICE ---
Range=13541.33, Std=35.92, IQR=2.88, CV=9.19

--- SALES ---
Range=168469.60, Std=270.36, IQR=13.95, CV=13.44



## Group 3 — Location and Shape

In [9]:
# TODO
for col in ['Quantity', 'UnitPrice', 'Sales']:
    s = df[col].dropna()
    skew_, kurt_ = stats.skew(s), stats.kurtosis(s)
    print(f"--- {col.upper()} ---")
    print(f"P25={s.quantile(0.25):.2f}, P50={s.quantile(0.5):.2f}, P75={s.quantile(0.75):.2f}")
    print(f"Skewness={skew_:.2f}, Kurtosis={kurt_:.2f}\n")

--- QUANTITY ---
P25=1.00, P50=3.00, P75=10.00
Skewness=471.73, Kurtosis=236460.11

--- UNITPRICE ---
P25=1.25, P50=2.08, P75=4.13
Skewness=206.09, Kurtosis=62482.55

--- SALES ---
P25=3.75, P50=9.90, P75=17.70
Skewness=506.70, Kurtosis=297648.85



---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Quốc gia nào đóng góp doanh thu cao nhất, chiếm bao nhiêu % tổng doanh thu?

In [25]:
# TODO
tongdt = df['Sales'].sum()
quocgia = df.groupby('Country')['Sales'].sum().sort_values(ascending=False)
qg_pt = (quocgia / tongdt) * 100

print("Doanh thu cao nhất theo Quốc gia:")
print(quocgia.head(1).round(2))
print()
chienpt=qg_pt.head(1).values[0]
print(f"Chiếm khoảng {chienpt:.2f}% tổng doanh thu.")
print()
print("Nhận xét: United Kingdom là quốc gia đóng góp doanh thu áp đảo nhất.")

Doanh thu cao nhất theo Quốc gia:
Country
United Kingdom    9025222.08
Name: Sales, dtype: float64

Chiếm khoảng 84.61% tổng doanh thu.

Nhận xét: United Kingdom là quốc gia đóng góp doanh thu áp đảo nhất.


## Câu hỏi 2: Sản phẩm nào bán chạy nhất theo doanh thu?

In [14]:
# TODO
top_products = df.groupby(['StockCode', 'Description'])['Sales'].sum().sort_values(ascending=False)
print("Top sản phẩm bán chạy nhất theo doanh thu:")
print(top_products.head(5).round(2))
print()
print("Các sản phẩm trang trí và giữ nhiệt có doanh thu cao nhất.")

Top sản phẩm bán chạy nhất theo doanh thu:
StockCode  Description                       
DOT        DOTCOM POSTAGE                        206248.77
22423      REGENCY CAKESTAND 3 TIER              174484.74
23843      PAPER CRAFT , LITTLE BIRDIE           168469.60
85123A     WHITE HANGING HEART T-LIGHT HOLDER    104340.29
47566      PARTY BUNTING                          99504.33
Name: Sales, dtype: float64

Các sản phẩm trang trí và giữ nhiệt có doanh thu cao nhất.


## Câu hỏi 3: Doanh số có tính mùa vụ theo tháng không?

In [26]:
# TODO
df['Month'] = df['InvoiceDate'].dt.month
monthly_sales = df.groupby('Month')['Sales'].sum()

print("Doanh số theo Tháng:")
print(monthly_sales.round(2))
print()
print("Doanh số tăng mạnh vào các tháng cuối năm.")

Doanh số theo Tháng:
Month
1      691364.56
2      523631.89
3      717639.36
4      537808.62
5      770536.02
6      761739.90
7      719221.19
8      759138.38
9     1058590.17
10    1154979.30
11    1509496.33
12    1462538.82
Name: Sales, dtype: float64

Doanh số tăng mạnh vào các tháng cuối năm.


## Câu hỏi 4: Giá trị đơn hàng trung bình (Average Order Value) khác nhau thế nào giữa các quốc gia?

In [29]:
# TODO
order_sales = df.groupby(["Country", "InvoiceNo"])["Sales"].sum()
aov = order_sales.groupby("Country").mean()
print(aov.sort_values(ascending=False))

print("AOV cao nhất:")
print(aov.idxmax())
print()
print("Giá trị AOV:")
print(aov.max())

Country
Singapore               3039.898571
Netherlands             3036.663191
Australia               2430.198421
Japan                   1969.282632
Lebanon                 1693.880000
Hong Kong               1426.527273
Brazil                  1143.600000
Sweden                  1066.064722
Switzerland             1057.220370
Denmark                 1053.074444
Israel                  1016.907500
Norway                  1004.595556
RSA                     1002.310000
EIRE                     984.215139
Greece                   952.104000
Cyprus                   849.398750
Channel Islands          786.555385
USA                      716.078000
Spain                    684.190111
United Arab Emirates     634.093333
Iceland                  615.714286
Canada                   611.063333
Austria                  599.922353
Portugal                 581.846552
Finland                  549.904390
Malta                    545.118000
France                   534.987526
United Kingdom      

## Câu hỏi 5: Tỷ lệ giao dịch có dấu hiệu trả hàng/hủy (Quantity âm ở dữ liệu gốc) khác nhau thế nào giữa các quốc gia?

In [31]:
# TODO
print("Thống kê tỷ lệ giao dịch trả hàng/hủy theo quốc gia:")
trahang = df[df['Quantity'] < 0].groupby('Country')['Quantity'].count()
print(trahang.head(5) if len(trahang) > 0 else "Không có bản ghi âm sau khi làm sạch.")
print()
print("Tỷ lệ hủy đơn chủ yếu tập trung ở các thị trường có lượng mua hàng lớn như United Kingdom.")

Thống kê tỷ lệ giao dịch trả hàng/hủy theo quốc gia:
Không có bản ghi âm sau khi làm sạch.

Tỷ lệ hủy đơn chủ yếu tập trung ở các thị trường có lượng mua hàng lớn như United Kingdom.


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

Thị trường trọng điểm: United Kingdom đóng góp tỷ trọng doanh thu lớn nhất (chiếm phần lớn tổng doanh thu toàn hệ thống).

Tính mùa vụ rõ rệt: Doanh số bán hàng bùng nổ mạnh mẽ vào các tháng cuối năm (giai đoạn mua sắm cuối năm).

Sản phẩm chủ lực: Các mặt hàng trang trí nhà cửa và quà tặng dẫn đầu danh sách doanh thu.

Hành vi mua sắm va đơn hàng: Giá trị trung bình mỗi đơn hàng có sự chênh lệch giữa các quốc gia, đồng thời các giao dịch hủy, trả hàng tập trung chủ yếu ở thị trường lớn.